In [1]:
# Imports
import os
import dotenv
from typing import TypedDict
import datetime
import json
import logging  # added logging

# Web Scraping
import requests

# 3rd Party APIs
import finnhub

# Image processing
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image

# Audio
from pydub import AudioSegment
from pydub.playback import play

# LLM APIs
from openai import OpenAI

# User interface
import gradio as gr

In [2]:
# Config

# Load environment variables
dotenv.load_dotenv()

# Logging configuration (idempotent & de-duplicated)
def _init_logger(name: str = "finance_chat") -> logging.Logger:
    logger = logging.getLogger(name)

    # Remove any duplicate finance_chat handlers (those we previously added)
    existing_fc_handlers = [h for h in logger.handlers if getattr(h, "_finance_chat", False)]
    if len(existing_fc_handlers) > 1:
        for h in existing_fc_handlers[1:]:
            logger.removeHandler(h)

    # Remove any non-tagged handlers to prevent duplicate logs in notebooks
    for h in list(logger.handlers):
        if not getattr(h, "_finance_chat", False):
            logger.removeHandler(h)

    level_name = os.getenv("FINANCE_CHAT_LOG_LEVEL", "INFO").upper()
    level = getattr(logging, level_name, logging.INFO)

    # If a tagged handler remains, just ensure level/propagate and return
    if any(getattr(h, "_finance_chat", False) for h in logger.handlers):
        logger.setLevel(level)
        logger.propagate = False
        return logger

    # Otherwise create and attach a single tagged handler
    handler = logging.StreamHandler()
    handler._finance_chat = True  # tag to recognize later
    formatter = logging.Formatter('[%(asctime)s] %(levelname)s %(name)s - %(message)s')
    handler.setFormatter(formatter)

    logger.addHandler(handler)
    logger.setLevel(level)
    logger.propagate = False
    return logger

logger = _init_logger()

# Finnhub Client
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")
finnhub_client = finnhub.Client(FINNHUB_API_KEY)

# LLM Client
openai = OpenAI()
SONAR_URL = "https://api.perplexity.ai/chat/completions"

# LLM models
OPEN_AI_MODEL = "gpt-5-nano"
OPEN_AI_IMAGE_MODEL = "dall-e-3"
OPEN_AI_AUDIO_MODEL = "tts-1"
GOOGLE_MODEL = "gemini-1.5-flash-latest"
SONAR_MODEL = "sonar-pro"

## Image Parameters
IMAGE_SIZE = "1024x1024"
N = 1
RESPONSE_FORMAT = "b64_json"
VOICE = "alloy"
AUDIO_FORMAT = "mp3"

# LLM Instructions
chat_system_message = """
You are a finance assistant that is able to summarize earnings call and report of the public companies and also
generate history charts of the selected finance metrics and desribe their trends.

Always be accurate. If you don't know the answer, say so.
"""

# Eearnings Call Agent
earnings_call_agent_system_message = """
You are a finance assistance that searches and summarizes the only last company earnings report and only from web search, 
for sales representative with focus on correlating facts with potential investments in AI/IT infrastructure returning the response 
in form of short statments/bullets. Ommit any side notes form search sources.

Be short and concise. If you cannot find a earnings call transcript
or report, say so.

Remove any links to the sources from the summary as this will be only text output in the textbox of the chatbot.

The example of the summary:
Company: Radiant Global Logistics Inc
* Very strong financial performance with 20% revenue growth year-over-year.
* Increased R&D expenses by 15%
* Increased CAPEX by 10%

Suggest to discuss where extra CAPEX and/or R&D investments can be made to improve the company's AI/IT infrastructure.

"""

earnings_call_agent_user_message = """
Summarize the most recent earnings call for {company_name} in year of {year}. 
"""


# Finance Metrics Agent
finance_metric_agent_system_message = """
You are a financial data assistant. Given a company name, a specific financial metric (e.g., revenue, net income, R&D, CAPEX), 
and a time window in years, extract a time series for that metric for the requested number of most recent years.

Always extract the data from finhub API using finhub_client.financials_reported function. Always call get_finance_metric function to get the data.
Parse the json response from finhub API to extract the relevant metric values for each fiscal quarter or year within the specified time window.

Return the result as a JSON object with the following structure:
{
  "x": ["FY21Q4", "FY22Q1", ..., "FY25Q4"],  // x-axis labels, each as fiscal year ex. FY22 or year-quarter ex. FY22Q1
  "y": [123.4, 150.2, ..., 210.0],           // y-axis values, one per x label rounded to 1 decimal place
  "x_label": "Fiscal Year & Quarter",        // Fiscal year if frquency is yearly or Fiscal Year & Quarter if frequency is quarterly
  "y_label": "Revenue (USD millions)"        // y_label should include the metric and its unit in parenthesis.
  "title": "Microsoft Corporation" // title should include the company name
}

Example for metric "revenue" for 4 years:
{
  "x": [ "FY22Q1", "FY22Q2", "FY22Q3", "FY22Q4", "FY23Q1", "FY23Q2", "FY23Q3", "FY23Q4", "FY24Q1", "FY24Q2", "FY24Q3", "FY24Q4", "FY25Q1", "FY25Q2", "FY25Q3", "FY25Q4"],
  "y": [120.5, 130.2, 128.7, 135.0, 140.1, 145.3, 150.2, 155.0, 160.4, 165.0, 170.2, 175.1, 180.0, 185.5, 190.2, 200.0],
  "x_label": "Fiscal Year & Quarter",
  "y_label": "Revenue (USD millions)" 
  "title": "Microsoft Corporation"
}

Reference below finhub API json response schema to extract:
The year, querter can be extracted from Filing secion using properties year and quarter.
The values can be extracted from FinancialLineItem using value property.
The metric can be inferred from FinancialLineItem concept or label property.
The unit can be extracted from unit property ofthe FinancialLineItem.

Json schema:
{
  "cik": "789019",
  "data": [
    {
      "accessNumber": "0000950170-25-061046",
      "symbol": "MSFT",
      "cik": "789019",
      "year": 2025,
      "quarter": 3,
      "form": "10-Q",
      "startDate": "2024-07-01 00:00:00",
      "endDate": "2025-03-31 00:00:00",
      "filedDate": "2025-04-30 00:00:00",
      "acceptedDate": "2025-04-30 16:08:52",
      "report": {
        "bs": [
          {
            "concept": "us-gaap_CashAndCashEquivalentsAtCarryingValue",
            "unit": "u_usd",
            "label": "Cash and Cash Equivalents, at Carrying Value, Total",
            "value": 28828000000.0
          },
          {
            "concept": "us-gaap_ShortTermInvestments",
            "unit": "u_usd",
            "label": "Short-Term Investments, Total",
            "value": 50790000000.0
          },
          {
            "concept": "us-gaap_CashCashEquivalentsAndShortTermInvestments",
            "unit": "u_usd",
            "label": "Total cash, cash equivalents, and short-term investments",
            "value": 79618000000
          },
          {
            "concept": "us-gaap_AccountsReceivableNetCurrent",
            "unit": "u_usd",
            "label": "Accounts Receivable, after Allowance for Credit Loss, Current, Total",
            "value": 51700000000
          },
          {
            "concept": "us-gaap_InventoryNet",
            "unit": "u_usd",
            "label": "Total",
            "value": 848000000.0
          },
          {
            "concept": "us-gaap_OtherAssetsCurrent",
            "unit": "u_usd",
            "label": "Other current assets",
            "value": 24478000000
          },
          {
            "concept": "us-gaap_AssetsCurrent",
            "unit": "u_usd",
            "label": "Total current assets",
            "value": 156644000000
          },
          {
            "concept": "us-gaap_PropertyPlantAndEquipmentNet",
            "unit": "u_usd",
            "label": "Property and equipment, net",
            "value": 183939000000
          },
          {
            "concept": "us-gaap_OperatingLeaseRightOfUseAsset",
            "unit": "u_usd",
            "label": "Operating lease right-of-use assets",
            "value": 24475000000.0
          },
          {
            "concept": "us-gaap_LongTermInvestments",
            "unit": "u_usd",
            "label": "Long-Term Investments, Total",
            "value": 16035000000.0
          },
          {
            "concept": "us-gaap_Goodwill",
            "unit": "u_usd",
            "label": "Goodwill",
            "value": 119329000000.0
          },
          {
            "concept": "us-gaap_FiniteLivedIntangibleAssetsNet",
            "unit": "u_usd",
            "label": "Finite-Lived Intangible Assets, Net, Ending Balance",
            "value": 23968000000.0
          },
          {
            "concept": "us-gaap_OtherAssetsNoncurrent",
            "unit": "u_usd",
            "label": "Other long-term assets",
            "value": 38234000000
          },
          {
            "concept": "us-gaap_Assets",
            "unit": "u_usd",
            "label": "Total assets",
            "value": 562624000000
          },
          {
            "concept": "us-gaap_AccountsPayableCurrent",
            "unit": "u_usd",
            "label": "Accounts Payable, Current, Total",
            "value": 26250000000
          },
          {
            "concept": "us-gaap_CommercialPaper",
            "unit": "u_usd",
            "label": "Commercial Paper",
            "value": 0.0
          },
          {
            "concept": "us-gaap_LongTermDebtCurrent",
            "unit": "u_usd",
            "label": "Current portion of long-term debt",
            "value": 2999000000.0
          },
          {
            "concept": "us-gaap_EmployeeRelatedLiabilitiesCurrent",
            "unit": "u_usd",
            "label": "Employee-related Liabilities, Current, Total",
            "value": 10579000000
          },
          {
            "concept": "us-gaap_AccruedIncomeTaxesCurrent",
            "unit": "u_usd",
            "label": "Short-term income taxes",
            "value": 6805000000
          },
          {
            "concept": "us-gaap_ContractWithCustomerLiabilityCurrent",
            "unit": "u_usd",
            "label": "Short-term unearned revenue",
            "value": 44636000000
          },
          {
            "concept": "us-gaap_OtherLiabilitiesCurrent",
            "unit": "u_usd",
            "label": "Other current liabilities",
            "value": 22937000000
          },
          {
            "concept": "us-gaap_LiabilitiesCurrent",
            "unit": "u_usd",
            "label": "Total current liabilities",
            "value": 114206000000
          },
          {
            "concept": "us-gaap_LongTermDebtNoncurrent",
            "unit": "u_usd",
            "label": "Long-Term Debt, Excluding Current Maturities, Total",
            "value": 39882000000.0
          },
          {
            "concept": "us-gaap_AccruedIncomeTaxesNoncurrent",
            "unit": "u_usd",
            "label": "Long-term income taxes",
            "value": 25061000000
          },
          {
            "concept": "us-gaap_ContractWithCustomerLiabilityNoncurrent",
            "unit": "u_usd",
            "label": "Long-term unearned revenue",
            "value": 2840000000
          },
          {
            "concept": "us-gaap_DeferredIncomeTaxLiabilitiesNet",
            "unit": "u_usd",
            "label": "Deferred income taxes",
            "value": 2522000000
          },
          {
            "concept": "us-gaap_OperatingLeaseLiabilityNoncurrent",
            "unit": "u_usd",
            "label": "Operating lease liabilities",
            "value": 17686000000
          },
          {
            "concept": "us-gaap_OtherLiabilitiesNoncurrent",
            "unit": "u_usd",
            "label": "Other long-term liabilities",
            "value": 38536000000
          },
          {
            "concept": "us-gaap_Liabilities",
            "unit": "u_usd",
            "label": "Total liabilities",
            "value": 240733000000
          },
          {
            "concept": "us-gaap_CommonStocksIncludingAdditionalPaidInCapital",
            "unit": "u_usd",
            "label": "Common Stocks, Including Additional Paid in Capital",
            "value": 106965000000
          },
          {
            "concept": "us-gaap_RetainedEarningsAccumulatedDeficit",
            "unit": "u_usd",
            "label": "Retained Earnings (Accumulated Deficit), Total",
            "value": 219759000000
          },
          {
            "concept": "us-gaap_AccumulatedOtherComprehensiveIncomeLossNetOfTax",
            "unit": "u_usd",
            "label": "Accumulated Other Comprehensive Income (Loss), Net of Tax, Total",
            "value": -4833000000
          },
          {
            "concept": "us-gaap_StockholdersEquity",
            "unit": "u_usd",
            "label": "Total stockholders\u2019 equity",
            "value": 321891000000.0
          },
          {
            "concept": "us-gaap_LiabilitiesAndStockholdersEquity",
            "unit": "u_usd",
            "label": "Total liabilities and stockholders\u2019 equity",
            "value": 562624000000
          }
        ],
        "ic": [
          {
            "concept": "us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax",
            "unit": "u_usd",
            "label": "Revenue",
            "value": 205283000000.0
          },
          {
            "concept": "us-gaap_CostOfGoodsAndServicesSold",
            "unit": "u_usd",
            "label": "Cost of Goods and Services Sold, Total",
            "value": 63817000000.0
          },
          {
            "concept": "us-gaap_GrossProfit",
            "unit": "u_usd",
            "label": "Gross margin",
            "value": 141466000000
          },
          {
            "concept": "us-gaap_ResearchAndDevelopmentExpense",
            "unit": "u_usd",
            "label": "Research and Development Expense, Total",
            "value": 23659000000
          },
          {
            "concept": "us-gaap_SellingAndMarketingExpense",
            "unit": "u_usd",
            "label": "Selling and Marketing Expense, Total",
            "value": 18369000000
          },
          {
            "concept": "us-gaap_GeneralAndAdministrativeExpense",
            "unit": "u_usd",
            "label": "General and Administrative Expense, Total",
            "value": 5233000000
          },
          {
            "concept": "us-gaap_OperatingIncomeLoss",
            "unit": "u_usd",
            "label": "Operating loss",
            "value": 94205000000.0
          },
          {
            "concept": "us-gaap_NonoperatingIncomeExpense",
            "unit": "u_usd",
            "label": "Other expense, net",
            "value": -3194000000.0
          },
          {
            "concept": "us-gaap_IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
            "unit": "u_usd",
            "label": "Income before income taxes",
            "value": 91011000000
          },
          {
            "concept": "us-gaap_IncomeTaxExpenseBenefit",
            "unit": "u_usd",
            "label": "Income Tax Expense (Benefit), Total",
            "value": 16412000000
          },
          {
            "concept": "us-gaap_NetIncomeLoss",
            "unit": "u_usd",
            "label": "Net income",
            "value": 74599000000.0
          },
          {
            "concept": "us-gaap_EarningsPerShareBasic",
            "unit": "u_unitedstatesofamericadollarsshare",
            "label": "Basic (A/B)",
            "value": 10.03
          },
          {
            "concept": "us-gaap_EarningsPerShareDiluted",
            "unit": "u_unitedstatesofamericadollarsshare",
            "label": "Diluted (A/C)",
            "value": 9.99
          },
          {
            "concept": "us-gaap_WeightedAverageNumberOfSharesOutstandingBasic",
            "unit": "u_shares",
            "label": "Weighted Average Number of Shares Outstanding, Basic, Total",
            "value": 7434000000.0
          },
          {
            "concept": "us-gaap_WeightedAverageNumberOfDilutedSharesOutstanding",
            "unit": "u_shares",
            "label": "Common stock and common stock equivalents (C)",
            "value": 7466000000.0
          },
          {
            "concept": "us-gaap_OtherComprehensiveIncomeLossCashFlowHedgeGainLossAfterReclassificationAndTax",
            "unit": "u_usd",
            "label": "Other Comprehensive Income (Loss), Cash Flow Hedge, Gain (Loss), after Reclassification and Tax, Total",
            "value": 4000000
          },
          {
            "concept": "us-gaap_OtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax",
            "unit": "u_usd",
            "label": "OCI, Debt Securities, Available-for-Sale, Gain (Loss), after Adjustment and Tax, Total",
            "value": 1130000000
          },
          {
            "concept": "us-gaap_OtherComprehensiveIncomeLossForeignCurrencyTransactionAndTranslationAdjustmentNetOfTax",
            "unit": "u_usd",
            "label": "Other Comprehensive Income (Loss), Foreign Currency Transaction and Translation Adjustment, Net of Tax, Total",
            "value": -377000000
          },
          {
            "concept": "us-gaap_OtherComprehensiveIncomeLossNetOfTaxPortionAttributableToParent",
            "unit": "u_usd",
            "label": "Other Comprehensive Income (Loss), Net of Tax, Portion Attributable to Parent",
            "value": 757000000
          },
          {
            "concept": "us-gaap_ComprehensiveIncomeNetOfTax",
            "unit": "u_usd",
            "label": "Comprehensive income",
            "value": 75356000000
          }
        ],
        "cf": [
          {
            "concept": "us-gaap_NetIncomeLoss",
            "unit": "u_usd",
            "label": "Net income",
            "value": 74599000000.0
          },
          {
            "concept": "msft_DepreciationAmortizationAndOther",
            "unit": "u_usd",
            "label": "Depreciation, amortization, and other",
            "value": 22950000000
          },
          {
            "concept": "us-gaap_ShareBasedCompensation",
            "unit": "u_usd",
            "label": "Share-Based Payment Arrangement, Noncash Expense, Total",
            "value": 8901000000
          },
          {
            "concept": "msft_GainLossOnInvestmentsAndDerivativeInstruments",
            "unit": "u_usd",
            "label": "Gain Loss On Investments And Derivative Instruments",
            "value": -553000000
          },
          {
            "concept": "us-gaap_DeferredIncomeTaxExpenseBenefit",
            "unit": "u_usd",
            "label": "Deferred Income Tax Expense (Benefit), Total",
            "value": -4835000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInAccountsReceivable",
            "unit": "u_usd",
            "label": "Accounts receivable",
            "value": -5598000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInInventories",
            "unit": "u_usd",
            "label": "Increase (Decrease) in Inventories, Total",
            "value": -390000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInOtherCurrentAssets",
            "unit": "u_usd",
            "label": "Other current assets",
            "value": -642000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInOtherNoncurrentAssets",
            "unit": "u_usd",
            "label": "Other long-term assets",
            "value": 3368000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInAccountsPayable",
            "unit": "u_usd",
            "label": "Increase (Decrease) in Accounts Payable, Total",
            "value": 1221000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInContractWithCustomerLiability",
            "unit": "u_usd",
            "label": "Unearned revenue",
            "value": -12923000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInAccruedIncomeTaxesPayable",
            "unit": "u_usd",
            "label": "Income taxes",
            "value": -1081000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInOtherCurrentLiabilities",
            "unit": "u_usd",
            "label": "Other current liabilities",
            "value": 576000000
          },
          {
            "concept": "us-gaap_IncreaseDecreaseInOtherNoncurrentLiabilities",
            "unit": "u_usd",
            "label": "Other long-term liabilities",
            "value": 292000000
          },
          {
            "concept": "us-gaap_NetCashProvidedByUsedInOperatingActivities",
            "unit": "u_usd",
            "label": "Net cash from operations",
            "value": 93515000000
          },
          {
            "concept": "us-gaap_ProceedsFromRepaymentsOfShortTermDebtMaturingInThreeMonthsOrLess",
            "unit": "u_usd",
            "label": "Proceeds from issuance (repayments) of debt, maturities of 90 days or less, net",
            "value": -5746000000
          },
          {
            "concept": "us-gaap_ProceedsFromDebtMaturingInMoreThanThreeMonths",
            "unit": "u_usd",
            "label": "Proceeds from Debt, Maturing in More than Three Months",
            "value": 0
          },
          {
            "concept": "us-gaap_RepaymentsOfDebtMaturingInMoreThanThreeMonths",
            "unit": "u_usd",
            "label": "Repayments of debt",
            "value": 3216000000
          },
          {
            "concept": "us-gaap_ProceedsFromIssuanceOfCommonStock",
            "unit": "u_usd",
            "label": "Common stock issued",
            "value": 1508000000
          },
          {
            "concept": "us-gaap_PaymentsForRepurchaseOfCommonStock",
            "unit": "u_usd",
            "label": "Common stock repurchased",
            "value": 13874000000
          },
          {
            "concept": "us-gaap_PaymentsOfDividendsCommonStock",
            "unit": "u_usd",
            "label": "Common stock cash dividends paid",
            "value": 17913000000
          },
          {
            "concept": "us-gaap_ProceedsFromPaymentsForOtherFinancingActivities",
            "unit": "u_usd",
            "label": "Other, net",
            "value": -1614000000
          },
          {
            "concept": "us-gaap_NetCashProvidedByUsedInFinancingActivities",
            "unit": "u_usd",
            "label": "Net Cash Provided by (Used in) Financing Activities",
            "value": -40855000000
          },
          {
            "concept": "us-gaap_PaymentsToAcquirePropertyPlantAndEquipment",
            "unit": "u_usd",
            "label": "Payments to Acquire Property, Plant, and Equipment, Total",
            "value": 47472000000
          },
          {
            "concept": "msft_AcquisitionsNetOfCashAcquiredAndPurchasesOfIntangibleAndOtherAssets",
            "unit": "u_usd",
            "label": "Acquisitions Net Of Cash Acquired And Purchases Of Intangible And Other Assets",
            "value": 4235000000
          },
          {
            "concept": "us-gaap_PaymentsToAcquireInvestments",
            "unit": "u_usd",
            "label": "Payments to Acquire Investments, Total",
            "value": 8144000000
          },
          {
            "concept": "us-gaap_ProceedsFromMaturitiesPrepaymentsAndCallsOfAvailableForSaleSecurities",
            "unit": "u_usd",
            "label": "Maturities of investments",
            "value": 11461000000
          },
          {
            "concept": "msft_ProceedsFromInvestments",
            "unit": "u_usd",
            "label": "Sales of investments",
            "value": 6688000000
          },
          {
            "concept": "us-gaap_PaymentsForProceedsFromOtherInvestingActivities",
            "unit": "u_usd",
            "label": "Other, net",
            "value": 325000000
          },
          {
            "concept": "us-gaap_NetCashProvidedByUsedInInvestingActivities",
            "unit": "u_usd",
            "label": "Net cash used in investing",
            "value": -42027000000
          },
          {
            "concept": "us-gaap_EffectOfExchangeRateOnCashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsIncludingDisposalGroupAndDiscontinuedOperations",
            "unit": "u_usd",
            "label": "Effect of Exchange Rate on Cash, Cash Equivalents, Restricted Cash, and Restricted Cash Equivalents, Including Disposal Group and Discontinued Operations, Total",
            "value": -120000000
          },
          {
            "concept": "us-gaap_CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect",
            "unit": "u_usd",
            "label": "Net change in cash and cash equivalents",
            "value": 10513000000
          }
        ]
      }
    },

If a value is missing, use null in the y array. Always organize the x and y arrays chronologically from oldest to most recent.
Return only valid JSON as described above.
"""
finance_metric_agent_user_message = """
Extract the time series for the company {company_name} for the metric {metric_name} on {frequency} basis and {time_window_in_years} years.
Return results  as a JSON object describe in system message. Avoid any extra text, explanations, or comments.
Return only valid JSON. Do not include any other text.
"""

# Tools

# Test variables
company_name = "Microsoft"
metric_name = "revenue"
time_window_in_years = 5
frequency = 'annual'  # 'annual' or 'quarterly'

In [3]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Sequence, Protocol, runtime_checkable, Any, Dict, List, Optional


# === LLM Client Strategy Interfaces ===
@runtime_checkable
class LLMClientStrategy(Protocol):
    """Strategy interface for different LLM providers."""

    def chat(self, model: str, messages: Sequence[Dict[str, str]]) -> Any: ...


@dataclass
class OpenAIChatClient:
    """Concrete strategy wrapping OpenAI chat API."""

    client: Any

    def chat(self, model: str, messages: Sequence[Dict[str, str]]) -> Any:
        return self.client.chat.completions.create(model=model, messages=list(messages))


@dataclass
class PerplexityChatClient:
    """Concrete strategy wrapping Perplexity (SONAR) HTTP API."""

    api_url: str
    api_key: str

    def chat(self, model: str, messages: Sequence[Dict[str, str]]) -> Any:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {"model": model, "messages": list(messages)}
        resp = requests.post(self.api_url, headers=headers, json=payload, timeout=60)
        resp.raise_for_status()
        return resp.json()


# === Template Method Base Agent ===
class BaseLLMAgent(ABC):
    """Template + Strategy based base agent.

    Responsibilities:
    - Defines the generate() template method.
    - Delegates provider differences to an injected strategy (LLMClientStrategy).
    - Child classes override hooks for system prompt, user messages, and parsing.
    """

    MODEL: str = SONAR_MODEL  # default

    def __init__(
        self,
        client_strategy: Optional[LLMClientStrategy] = None,
        model_name: Optional[str] = None,
    ):
        self.model_name = model_name or self.MODEL
        # Default strategy: Perplexity if SONAR model else OpenAI
        if client_strategy:
            self._client = client_strategy
        else:
            if self.model_name.startswith("sonar"):
                self._client = PerplexityChatClient(
                    api_url=SONAR_URL, api_key=os.getenv("SONAR_API_KEY", "")
                )
            else:
                self._client = OpenAIChatClient(client=openai)

    # --- Hooks ---
    @property
    @abstractmethod
    def system_prompt(self) -> str: ...

    @abstractmethod
    def build_user_messages(self, *args, **kwargs) -> List[Dict[str, str]]:
        """Return a list of user (and optionally assistant) messages excluding system."""
        ...

    def build_messages(self, *args, **kwargs) -> List[Dict[str, str]]:
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(self.build_user_messages(*args, **kwargs))
        return messages

    def parse_response(self, raw: Any) -> str:
        """Default parser accommodating both OpenAI + Perplexity schemas."""
        # OpenAI returns object with .choices[0].message.content; Perplexity returns JSON
        try:
            # Attempt OpenAI style
            return raw.choices[0].message.content  # type: ignore[attr-defined]
        except Exception:
            # Attempt Perplexity style dict
            if isinstance(raw, dict):
                return raw.get("choices", [{}])[0].get("message", {}).get("content", "")
            return str(raw)

    # --- Template Method ---
    def generate(self, *args, **kwargs) -> str:
        messages = self.build_messages(*args, **kwargs)
        raw = self._client.chat(self.model_name, messages)
        return self.parse_response(raw)


# === Concrete Agents ===
class EarningsCallAgent(BaseLLMAgent):
    MODEL = SONAR_MODEL

    @property
    def system_prompt(self) -> str:
        return earnings_call_agent_system_message

    def build_user_messages(self, company_name: str) -> List[Dict[str, str]]:
        prompt = earnings_call_agent_user_message.format(
            company_name=company_name, year=datetime.datetime.now().year
        )
        return [{"role": "user", "content": prompt}]

    # Convenience wrapper
    def summarize_earnings_call(self, company_name: str) -> str:
        return self.generate(company_name)


class FinanceMetricsAgent(BaseLLMAgent):
    MODEL = OPEN_AI_MODEL

    @property
    def system_prompt(self) -> str:
        return finance_metric_agent_system_message

    def build_user_messages(
        self,
        company_name: str,
        metric_name: str,
        frequency: str,
        time_window_in_years: int,
        raw_data: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, str]]:
        base = finance_metric_agent_user_message.format(
            company_name=company_name,
            metric_name=metric_name,
            frequency=frequency,
            time_window_in_years=time_window_in_years,
        )
        msgs = [{"role": "user", "content": base}]
        if raw_data:
            data_json = json.dumps(raw_data, default=str)
            msgs.append({"role": "user", "content": data_json})
        return msgs

    def summarize_financial_metrics(
        self,
        company_name: str,
        stock_symbol: str,
        metric_name: str,
        frequency: str = "annual",
        time_window_in_years: int = 5,
    ) -> str:
        raw = self._get_finance_metric(company_name, stock_symbol, frequency)
        summary = self.generate(
            company_name, metric_name, frequency, time_window_in_years, raw
        )
        try:
            logger.info(
                f"agent.metrics_summary company={company_name} symbol={stock_symbol} metric={metric_name} freq={frequency} years={time_window_in_years} -> {summary}"
            )
        except Exception as e:
            logger.warning(f"agent.metrics_summary.log_error: {e}")
        # Quality at the source: validate/repair JSON structure (x, y, x_label, y_label)
        fixed = self._coerce_metrics_json(
            summary, company_name, metric_name, frequency, time_window_in_years
        )
        return fixed

    def _is_valid_metrics_payload(self, obj: Any) -> bool:
        if not isinstance(obj, dict):
            return False
        x = obj.get("x")
        y = obj.get("y")
        xl = obj.get("x_label")
        yl = obj.get("y_label")
        title = obj.get("title")
        if (
            not isinstance(x, list)
            or not isinstance(y, list)
            or not isinstance(xl, str)
            or not isinstance(yl, str)
            or not isinstance(title, str)
        ):
            return False
        if len(x) != len(y) or len(x) == 0:
            return False
        # Ensure y contains numeric-like values
        try:
            [float(v) for v in y]
        except Exception:
            return False
        return True

    def _coerce_metrics_json(
        self,
        text: str,
        company_name: str,
        metric_name: str,
        frequency: str,
        time_window_in_years: int,
    ) -> str:
        # First, try strict JSON parsing
        def _strip_fences(t: str) -> str:
            t = t.strip()
            if t.startswith("```") and t.endswith("```"):
                t = t.strip("`")
                # crude fence removal fallback
            return t

        candidate = _strip_fences(text)
        try:
            obj = json.loads(candidate)
            if self._is_valid_metrics_payload(obj):
                return json.dumps(obj)
        except Exception:
            pass

        # Ask the LLM to repair: best-practice JSON repair prompt
        repair_msgs = [
            {
                "role": "system",
                "content": "You are a strict JSON reformatter. Return only valid JSON with keys x (list of strings), y (list of numbers), x_label (string), y_label (string), title (string that contains the company name). No extra text.",
            },
            {
                "role": "user",
                "content": f"Given the company {company_name}, metric {metric_name}, frequency {frequency}, time window {time_window_in_years} years, reformat the following into the required JSON schema.Content:{text}",
            },
        ]
        try:
            raw = self._client.chat(self.model_name, repair_msgs)
            fixed_text = self.parse_response(raw)
            obj = json.loads(fixed_text)
            if self._is_valid_metrics_payload(obj):
                return json.dumps(obj)
        except Exception as e:
            logger.warning(f"metrics_json.repair_failed: {e}")

        # As a final fallback, return original text so UI still shows a response
        return text

    def _get_finance_metric(
        self, company_name: str, stock_symbol: str, frequency: str = "annual"
    ) -> Dict[str, Any]:
        """Thin wrapper over finnhub financials_reported.

        Parameters
        ----------
        company_name : str
            Company name (currently unused but kept for future enrichment / logging).
        stock_symbol : str
            Ticker symbol accepted by Finnhub.
        frequency : str
            'annual' or 'quarterly'.
        """
        if frequency not in {"annual", "quarterly"}:
            raise ValueError("frequency must be 'annual' or 'quarterly'")
        financials_json = finnhub_client.financials_reported(symbol=stock_symbol, freq=frequency)
        # save datato json file for debugging
        with open(f"{company_name}_{stock_symbol}_financials_{frequency}.json", "w") as f:
            json.dump(financials_json, f, indent=2, default=str)
        return financials_json

In [4]:

# === Image Generation ===
class ImageGenerator:
    """Utility to produce matplotlib charts as PIL.Images from JSON-like time series."""

    # Static defaults (can be overridden in __init__)
    FIGSIZE = (7, 4)
    LINE_COLOR = "#1f77b4"
    LINE_WIDTH = 2
    MARKER = "o"
    GRID_ALPHA = 0.3
    TITLE_FMT = "{y_label} Trend"
    XLABEL = "Period"

    def __init__(self, figsize=None, line_color=None, line_width=None, marker=None, grid_alpha=None, title_fmt=None, xlabel=None):
        self.figsize = figsize or self.FIGSIZE
        self.line_color = line_color or self.LINE_COLOR
        self.line_width = line_width or self.LINE_WIDTH
        self.marker = marker or self.MARKER
        self.grid_alpha = grid_alpha or self.GRID_ALPHA
        self.title_fmt = title_fmt or self.TITLE_FMT
        self.xlabel = xlabel or self.XLABEL

    def create_matlotlib_chart(self, summary_json_text: str, company_name: Optional[str] = None) -> Image:
        """Create a trend chart from standardized JSON and return a PIL.Image via in-memory buffer.
        Expects JSON: { x: list[str], y: list[number], x_label: str, y_label: str, title: str }.
        The title is taken from 'title' if present, otherwise falls back to company_name.
        """
        # Strip code fences if present
        t = (summary_json_text or "").strip()
        if t.startswith('```') and t.endswith('```'):
            t = t.strip('`')
        obj = json.loads(t)
        if not isinstance(obj, dict):
            raise ValueError("Metrics payload must be a JSON object.")
        x = obj.get('x'); y = obj.get('y'); xl = obj.get('x_label'); yl = obj.get('y_label'); ti = obj.get('title')
        if not isinstance(x, list) or not isinstance(y, list) or not isinstance(xl, str) or not isinstance(yl, str):
            raise ValueError("Metrics payload missing required keys or wrong types.")
        if len(x) != len(y) or len(x) == 0:
            raise ValueError("x and y must be non-empty lists of equal length.")
        y_vals = [float(v) for v in y]

        plt.figure(figsize=self.figsize)
        plt.plot(x, y_vals, marker=self.marker, linewidth=self.line_width, color=self.line_color)
        # Title (prefer JSON 'title', else provided company name)
        title_str = ti if isinstance(ti, str) and ti.strip() else (company_name or None)
        if title_str:
            plt.title(title_str)
        plt.xlabel(xl)
        plt.ylabel(yl)
        plt.grid(True, alpha=self.grid_alpha)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        buf = BytesIO(); plt.savefig(buf, format="png"); plt.close(); buf.seek(0)
        return Image.open(buf)

def fn_get_finance_metrics(company_name: str, stock_symbol: str, metric_name: str, frequency: str = "annual", time_window_in_years: int = 5):
    """Return (text_json, PIL.Image|None) for the requested metric.
    Uses FinanceMetricsAgent and ImageGenerator.
    """
    agent = FinanceMetricsAgent()
    text = agent.summarize_financial_metrics(company_name, stock_symbol, metric_name, frequency, time_window_in_years)
    img = None
    try:
        img = ImageGenerator().create_matlotlib_chart(text, company_name=company_name)
    except Exception as e:
        logger.warning(f"chart.generate failed: {e}")
    return text, img

def fn_get_finance_metric_chart(summary_json_text: str, company_name: Optional[str] = None) -> Image:
    """Thin wrapper over ImageGenerator for convenience."""
    return ImageGenerator().create_matlotlib_chart(summary_json_text, company_name=company_name)

def fn_get_earnings_call_summary(company_name: str):
    """Return (text_summary, None) for latest earnings call."""
    agent = EarningsCallAgent()
    return agent.summarize_earnings_call(company_name), None

# === Tool Specifications ===
fn_get_finance_metrics_spec = {
    "name": "fn_get_finance_metrics",
    "description": "Get structured financial metric time series for a public company using the FinanceMetricsAgent. Returns JSON time series for the requested company, metric, frequency and time window in years.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "The name of the company to get financial metrics for."},
            "stock_symbol": {"type": "string", "description": "The stock symbol of the company, search on internet by company name if not provided."},
            "metric_name": {"type": "string", "description": "The financial metric to extract (e.g., revenue, net income, research and development expenses)."},
            "frequency": {"type": "string", "enum": ["quarterly", "annual"], "default": "annual", "description": "Frequency of the financial report (quarterly or annual)."},
            "time_window_in_years": {"type": "integer", "default": 5, "minimum": 1, "description": "How many most recent years to include in the time series."},
        },
        "required": ["company_name", "stock_symbol", "metric_name"],
        "additionalProperties": False,
    },
}

fn_get_earnings_call_summary_spec = {
    "name": "fn_get_earnings_call_summary",
    "description": "Summarize the most recent earnings call for a public company using the EarningsCallAgent.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "The name of the company to summarize the earnings call for."}
        },
        "required": ["company_name"],
        "additionalProperties": False,
    },
}

# Aggregate tool definitions for the OpenAI chat API
tools = [
    {"type": "function", "function": fn_get_finance_metrics_spec},
    {"type": "function", "function": fn_get_earnings_call_summary_spec},
]

In [5]:
def talker(message):
    response = openai.audio.speech.create(
        model=OPEN_AI_AUDIO_MODEL,
        voice=VOICE,
        input=message
    )

    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format=AUDIO_FORMAT)
    play(audio)

In [6]:
class ChatMessage(TypedDict):
    role: str
    content: str

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    name = tool_call.function.name
    logger.info(f"tool.dispatch name={name} args={arguments}")

    img = None
    if name == "fn_get_finance_metrics":
        result_text, img = fn_get_finance_metrics(
            arguments.get("company_name"),
            arguments.get("stock_symbol"),
            arguments.get("metric_name"),
            arguments.get("frequency", "annual"),
            arguments.get("time_window_in_years", 5)
        )
    elif name == "fn_get_earnings_call_summary":
        result_text, img = fn_get_earnings_call_summary(arguments.get("company_name"))
    else:
        logger.warning(f"tool.unknown name={name}")
        result_text = ""
    logger.info(f"tool.complete name={name} bytes={len(result_text) if isinstance(result_text,str) else 'n/a'}")
    tool_msg = {
        "role": "tool",
        "content": result_text,
        "tool_call_id": tool_call.id,
        "name": name,
    }
    return tool_msg, img

def chat(history: List[ChatMessage]):
    messages = [{"role": "system", "content": chat_system_message}] + history
    last_user = next((m['content'] for m in reversed(messages) if m['role']=='user'), None)
    logger.info(f"chat.start msg_count={len(messages)} last_user={last_user!r}")
    out_img = None

    response = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages, tools=tools)
    choice = response.choices[0]
    if choice.finish_reason == "tool_calls":
        message = choice.message
        tc = message.tool_calls[0]
        logger.info(f"chat.tool_request name={tc.function.name}")
        tool_msg, img = handle_tool_call(message)
        if img is not None:
            out_img = img
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }],
        })
        messages.append(tool_msg)
        response = openai.chat.completions.create(model=OPEN_AI_MODEL, messages=messages)
        choice = response.choices[0]
        logger.info(f"chat.tool_round_complete name={tc.function.name}")

    reply = choice.message.content
    history.append({"role": "assistant", "content": reply})
    logger.info("chat.complete")
    cleaned = [{"role": m.get("role"), "content": (m.get("content") or "")} for m in history if isinstance(m, dict) and m.get("role") in ("user", "assistant")]

    talker(reply)
    return cleaned, (out_img if out_img is not None else gr.update())

In [7]:
# # Example usage to demonstrate the chat function with the fn_get_finance_metrics tool
# user_message = f"Please provide me revenue report for {company_name} on {frequency} basis."
# history = [{"role": "user", "content": user_message}]
# history = chat(history)
# print(history)

In [8]:
# Example usage to demonstrate the chat function with the new fn_get_earnings_call_summary tool
# user_message = f"Please provide me earnings call summary for {company_name}."
# history = [{"role": "user", "content": user_message}]
# history = chat(history)
# print(history)

In [ ]:
# Gradio UI with chat + image + input
ALLOWED_ROLES = {"user", "assistant"}
def _sanitize_messages(msgs):
    cleaned = []
    for m in msgs:
        if not isinstance(m, dict):
            continue
        role = m.get("role"); content = m.get("content")
        if role in ALLOWED_ROLES and isinstance(content, (str, type(None))):
            cleaned.append({"role": role, "content": content or ""})
    return cleaned

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        message = (message or "").strip()
        if not message:
            return "", history
        logger.info(f"UI received prompt: {message}")
        history = list(history or [])
        history += [{"role": "user", "content": message}]
        return "", _sanitize_messages(history)

    entry.submit(fn=do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        fn=chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

    clear.click(fn=lambda: [], inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True, inline=False, share=False)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[2025-09-14 14:51:35,014] INFO finance_chat - UI received prompt: Summarize last Microsoft Earnings call
[2025-09-14 14:51:35,114] INFO finance_chat - chat.start msg_count=2 last_user='Summarize last Microsoft Earnings call'
[2025-09-14 14:51:37,842] INFO finance_chat - chat.tool_request name=fn_get_earnings_call_summary
[2025-09-14 14:51:37,844] INFO finance_chat - tool.dispatch name=fn_get_earnings_call_summary args={'company_name': 'Microsoft'}
[2025-09-14 14:51:45,732] INFO finance_chat - tool.complete name=fn_get_earnings_call_summary bytes=908
[2025-09-14 14:51:55,754] INFO finance_chat - chat.tool_round_complete name=fn_get_earnings_call_summary
[2025-09-14 14:51:55,755] INFO finance_chat - chat.complete
Input #0, wav, from '/tmp/tmpqpc9wc_o.wav':   0KB sq=    0B f=0/0   
  Duration: 00:01:27.00, bitrate: 384 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 24000 Hz, 1 channels, s16, 384 kb/s
